# Fiddler Chart Metric Pull

This notebook demonstrates how to:
1. **Query** an existing chart from a Fiddler dashboard
2. **Record** the chart's metadata (metric, time bins, time window, title, etc.)
3. **Pull** the underlying metric values and save them to a CSV file

### Prerequisites
- A Fiddler instance URL and API token
- A dashboard ID (UUID) from your Fiddler instance

### API Endpoints Used
| Endpoint | Method | Purpose |
|----------|--------|---------|
| `/v2/dashboards/{dashboard_id}` | GET | Fetch dashboard layout and chart IDs |
| `/v3/charts/{chart_id}` | GET | Fetch full chart definition and metadata |
| `/v3/queries` | POST | Pull monitoring metric time-series data |

> **Note:** The dashboard and chart APIs are unofficial and may change without notice.

---
## Setup and Configuration

In [1]:
import requests
import uuid
import json
import csv
import os
from datetime import datetime
from pprint import pprint

In [2]:
# =============================================================================
# CONFIGURATION - Fill in your Fiddler instance details
# =============================================================================

# Fiddler instance URL (e.g. "https://acme.cloud.fiddler.ai")
BASE_URL = ""

# Fiddler API key
TOKEN = ""

# Dashboard ID (UUID) - find this in the URL when viewing a dashboard in the Fiddler UI
# e.g. https://acme.cloud.fiddler.ai/dashboards/xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx
DASHBOARD_ID = ""

# Request headers
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# Output directory for CSV files
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Base URL: {BASE_URL}")
print(f"Dashboard ID: {DASHBOARD_ID}")
print(f"Output directory: {OUTPUT_DIR}/")

Base URL: https://demo.fiddler.ai
Dashboard ID: dbd97cc9-3cfb-4d22-8159-a025ced9b455
Output directory: output/


---
## Step 1: Query the Dashboard for Charts

Fetch the dashboard definition to discover all charts it contains, then retrieve the full definition for each chart.

In [3]:
# Fetch the dashboard to get the list of chart IDs
dashboard_url = f"{BASE_URL}/v2/dashboards/{DASHBOARD_ID}"

response = requests.get(dashboard_url, headers=HEADERS)
response.raise_for_status()
dashboard_response = response.json()

# The dashboard data may be nested under a 'data' key
dashboard_data = dashboard_response.get("data", dashboard_response)

# Extract chart IDs from the dashboard layouts
layouts = dashboard_data.get("layouts", [])
chart_ids = [layout["chart_uuid"] for layout in layouts if layout.get("chart_uuid")]

print(f"Dashboard: {dashboard_data.get('title', dashboard_data.get('name', 'Untitled'))}")
print(f"Found {len(chart_ids)} charts in dashboard")
for i, cid in enumerate(chart_ids):
    print(f"  [{i}] {cid}")

Dashboard: Bank Churn Business Intelligence Dashboard
Found 13 charts in dashboard
  [0] e19524ee-bea9-4589-b5d5-d033d44613d9
  [1] 73f26a2c-7e72-4150-9c0d-bf63afdc4722
  [2] 9c32ae13-0a1f-43b0-9617-f7c2ef419adb
  [3] c41d24d7-8e83-4845-b3d8-d436a2dd5c91
  [4] 937959a8-7566-424a-a54a-fa13990fdd55
  [5] cf70291b-2e24-4f7a-9ba8-9ce846bf47b9
  [6] 4e950e96-8de1-41f8-8f8c-7fe10c4a8c96
  [7] ac4bfc9c-9c2e-4755-805c-0999d61691bf
  [8] ccdbd4fb-341f-4f43-8fd3-a806fee72089
  [9] 65b1b896-a6c8-42b1-867c-bc3ed4e8d2f9
  [10] b5ff1ca9-f4a4-48f1-972c-b41031d2ac41
  [11] eaa1afe4-aac2-4ecb-9ce8-a8b157fd71e5
  [12] 03c3e727-c485-4959-9201-14faf171ca67


In [4]:
# Fetch the full definition for each chart
charts = []

for chart_id in chart_ids:
    chart_url = f"{BASE_URL}/v3/charts/{chart_id}"
    resp = requests.get(chart_url, headers=HEADERS)
    resp.raise_for_status()
    chart_resp = resp.json()
    chart = chart_resp.get("data", chart_resp)
    charts.append(chart)

print(f"\nRetrieved {len(charts)} chart definitions:")
for i, chart in enumerate(charts):
    print(f"  [{i}] {chart.get('title', 'Untitled')} (type: {chart.get('query_type', 'N/A')})")


Retrieved 13 chart definitions:
  [0] Model Performance - Monthly (type: ANALYTICS)
  [1] Revenue Impact By State (type: MONITORING)
  [2] Prediction Drift (type: MONITORING)
  [3] Accuracy By State (type: MONITORING)
  [4] Churn Probability - Gender (type: MONITORING)
  [5] Churn Probability - Geography (type: MONITORING)
  [6] Group Benefit - Gender (type: MONITORING)
  [7] Data Integrity Metrics - Bank Churn (type: MONITORING)
  [8] Accuracy (type: MONITORING)
  [9] Accuracy for California Users (type: MONITORING)
  [10] Decision Distribution  (type: ANALYTICS)
  [11] Decision Volume Tracking (type: MONITORING)
  [12] Confusion Matrix for California Customers (type: ANALYTICS)


### Select a Chart

Set `CHART_INDEX` below to choose which chart to work with. You can also re-run the cells below for different charts.

In [5]:
# Select which chart to pull metrics for (0-indexed from the list above)
CHART_INDEX = 2

selected_chart = charts[CHART_INDEX]
print(f"Selected chart: {selected_chart.get('title', 'Untitled')}")

Selected chart: Prediction Drift


---
## Step 2: Extract and Record Chart Metadata

Parse the chart definition to extract key metadata: the metric being plotted, time bin size, time window, model reference, visualization type, and more.

In [6]:
def extract_chart_metadata(chart):
    """Extract key metadata from a chart definition.

    Handles both MONITORING and ANALYTICS chart types.

    Returns a dict with:
        - title, chart_id, query_type, description
        - For MONITORING: bin_size, time_zone, time_label, time_range,
          and per-query details (metric, model_id, viz_type, columns, etc.)
        - For ANALYTICS: model_id, env_type, and per-metric details
    """
    meta = {
        "title": chart.get("title", "Untitled"),
        "chart_id": chart.get("id"),
        "query_type": chart.get("query_type"),
        "description": chart.get("description", ""),
        "options": chart.get("options", {}),
    }

    data_source = chart.get("data_source", {})
    ds_query_type = data_source.get("query_type", chart.get("query_type"))
    meta["data_source_query_type"] = ds_query_type

    if ds_query_type == "MONITORING":
        # Extract time filter settings
        filters = data_source.get("filters", {})
        meta["bin_size"] = filters.get("bin_size")
        meta["time_zone"] = filters.get("time_zone")
        meta["time_label"] = filters.get("time_label")
        time_range = filters.get("time_range", {})
        meta["start_time"] = time_range.get("start_time")
        meta["end_time"] = time_range.get("end_time")

        # Extract per-query details
        queries = data_source.get("queries", [])
        meta["queries"] = []
        for q in queries:
            query_meta = {
                "metric": q.get("metric"),
                "metric_type": q.get("metric_type"),
                "model_id": q.get("model", {}).get("id") if isinstance(q.get("model"), dict) else q.get("model_id"),
                "model_name": q.get("model", {}).get("name") if isinstance(q.get("model"), dict) else q.get("model_name"),
                "viz_type": q.get("viz_type"),
                "columns": q.get("columns", []),
                "baseline_id": q.get("baseline_id", ""),
                "segment": q.get("segment", {}),
                "query_key": q.get("query_key"),
            }
            meta["queries"].append(query_meta)

        # Also extract project_id from the chart or data_source
        project = chart.get("project", {})
        if isinstance(project, dict):
            meta["project_id"] = project.get("id")
            meta["project_name"] = project.get("name")
        else:
            meta["project_id"] = chart.get("project_id")
            meta["project_name"] = None

    elif ds_query_type == "ANALYTICS":
        # ANALYTICS charts store config differently
        model_ref = data_source.get("model", {})
        meta["model_id"] = model_ref.get("id") if isinstance(model_ref, dict) else None
        meta["model_name"] = model_ref.get("name") if isinstance(model_ref, dict) else None
        meta["env_type"] = data_source.get("env_type")

        payload = data_source.get("payload", {})
        meta["time_range"] = payload.get("time_range", {})
        meta["start_time"] = meta["time_range"].get("start_time")
        meta["end_time"] = meta["time_range"].get("end_time")

        meta["metrics"] = payload.get("metrics", [])

        project = chart.get("project", {})
        if isinstance(project, dict):
            meta["project_id"] = project.get("id")
            meta["project_name"] = project.get("name")
        else:
            meta["project_id"] = chart.get("project_id")
            meta["project_name"] = None

    elif ds_query_type in ("GEN_AI_MONITORING", "GEN_AI_METRICS"):
        # Agentic / GenAI charts
        time_filters = data_source.get("time_filters", {})
        meta["bin_size"] = time_filters.get("bin_size")
        meta["time_zone"] = time_filters.get("time_zone")
        time_range = time_filters.get("time_range", {})
        meta["start_time"] = time_range.get("start_time")
        meta["end_time"] = time_range.get("end_time")

        queries = data_source.get("queries", [])
        meta["queries"] = []
        for q in queries:
            query_meta = {
                "metric_name": q.get("metric_name"),
                "metric_source": q.get("metric_source"),
                "aggregation": q.get("aggregation"),
                "query_scope": q.get("query_scope"),
                "application_id": q.get("application_id"),
                "viz_type": q.get("viz_type"),
                "filters": q.get("filters", {}),
            }
            meta["queries"].append(query_meta)

        project = chart.get("project", {})
        if isinstance(project, dict):
            meta["project_id"] = project.get("id")
        else:
            meta["project_id"] = chart.get("project_id")

    return meta


metadata = extract_chart_metadata(selected_chart)

print("=" * 60)
print("CHART METADATA")
print("=" * 60)
print(f"  Title:       {metadata['title']}")
print(f"  Chart ID:    {metadata['chart_id']}")
print(f"  Query Type:  {metadata.get('data_source_query_type')}")
print(f"  Description: {metadata.get('description', '')}")
print(f"  Project ID:  {metadata.get('project_id')}")
print(f"  Start Time:  {metadata.get('start_time')}")
print(f"  End Time:    {metadata.get('end_time')}")

if metadata.get("data_source_query_type") == "MONITORING":
    print(f"  Bin Size:    {metadata.get('bin_size')}")
    print(f"  Time Zone:   {metadata.get('time_zone')}")
    print(f"  Time Label:  {metadata.get('time_label')}")
    print(f"\n  Queries ({len(metadata.get('queries', []))})")
    for i, q in enumerate(metadata.get("queries", [])):
        print(f"    [{i}] metric={q['metric']}, model={q.get('model_name') or q.get('model_id')}, viz={q['viz_type']}")
        if q.get("columns"):
            print(f"        columns={q['columns']}")

elif metadata.get("data_source_query_type") == "ANALYTICS":
    print(f"  Model:       {metadata.get('model_name')} ({metadata.get('model_id')})")
    print(f"  Env Type:    {metadata.get('env_type')}")
    print(f"\n  Metrics ({len(metadata.get('metrics', []))})")
    for i, m in enumerate(metadata.get("metrics", [])):
        print(f"    [{i}] id={m.get('id')}, baseline={m.get('baseline_id', 'N/A')}")

elif metadata.get("data_source_query_type") in ("GEN_AI_MONITORING", "GEN_AI_METRICS"):
    print(f"  Bin Size:    {metadata.get('bin_size')}")
    print(f"  Time Zone:   {metadata.get('time_zone')}")
    print(f"\n  Queries ({len(metadata.get('queries', []))})")
    for i, q in enumerate(metadata.get("queries", [])):
        print(f"    [{i}] metric={q['metric_name']}, agg={q.get('aggregation')}, scope={q.get('query_scope')}")

CHART METADATA
  Title:       Prediction Drift
  Chart ID:    9c32ae13-0a1f-43b0-9617-f7c2ef419adb
  Query Type:  MONITORING
  Description: Model Output Drift (JSD))
  Project ID:  1eaf9070-af0c-4d50-a3c4-1f7968bd0574
  Start Time:  2025-09-17T00:00:00
  End Time:    2025-10-16T23:59:59
  Bin Size:    Day
  Time Zone:   UTC
  Time Label:  30d

  Queries (3)
    [0] metric=jsd, model=churn_classifier, viz=line
        columns=['probability_churn']
    [1] metric=jsd, model=churn_classifier, viz=line
        columns=['probability_churn']
    [2] metric=jsd, model=churn_classifier, viz=line
        columns=['probability_churn']


In [ ]:
# Optionally: inspect the full raw chart definition
print(json.dumps(selected_chart, indent=2, default=str))

---
## Step 3: Pull Metric Values Using Chart Metadata

Reconstruct the query payload from the chart metadata and call the `/v3/queries` endpoint to retrieve the time-series data that powers the chart.

In [10]:
def build_monitoring_payload(metadata):
    """Build a /v3/queries payload from MONITORING chart metadata.

    Returns:
        (payload, query_key_map) where query_key_map maps each new
        query_key (used in the payload) back to the index in
        metadata['queries'] so the parser can resolve metric/model labels.
    """
    queries = []
    query_key_map = {}  # new_query_key -> query index
    for i, q in enumerate(metadata.get("queries", [])):
        new_key = str(uuid.uuid4())
        query_key_map[new_key] = i
        query = {
            "metric": q["metric"],
            "model_id": q["model_id"],
            "query_key": new_key,
            "viz_type": q.get("viz_type", "line"),
            "columns": q.get("columns", []),
            "baseline_id": q.get("baseline_id", ""),
            "segment": q.get("segment", {}),
        }
        queries.append(query)

    payload = {
        "project_id": metadata["project_id"],
        "query_type": "MONITORING",
        "filters": {
            "bin_size": metadata.get("bin_size", "Day"),
            "time_zone": metadata.get("time_zone", "UTC"),
            "time_range": {
                "start_time": metadata["start_time"],
                "end_time": metadata["end_time"],
            },
        },
        "queries": queries,
    }

    # Include time_label if present (e.g. "30d", "7d")
    if metadata.get("time_label"):
        payload["filters"]["time_label"] = metadata["time_label"]

    return payload, query_key_map


def build_analytics_payload(metadata):
    """Build a /v3/analytics/metrics payload from ANALYTICS chart metadata."""
    payload = {
        "model_id": metadata["model_id"],
        "env_type": metadata.get("env_type", "PRODUCTION"),
        "metrics": metadata.get("metrics", []),
        "time_range": {
            "start_time": metadata["start_time"],
            "end_time": metadata["end_time"],
        },
    }
    return payload


def build_genai_monitoring_payload(metadata):
    """Build a /v3/queries payload from GEN_AI_MONITORING chart metadata.

    Returns:
        (payload, query_key_map) where query_key_map maps each new
        query_key back to the index in metadata['queries'].
    """
    queries = []
    query_key_map = {}  # new_query_key -> query index
    for i, q in enumerate(metadata.get("queries", [])):
        new_key = str(uuid.uuid4())
        query_key_map[new_key] = i
        query = {
            "application_id": q["application_id"],
            "metric_name": q["metric_name"],
            "metric_source": q.get("metric_source", "raw_data"),
            "query_key": new_key,
            "query_type": metadata.get("data_source_query_type"),
            "query_scope": q.get("query_scope", "SPAN"),
            "aggregation": q.get("aggregation", "count"),
            "filters": q.get("filters", {}),
            "viz_type": q.get("viz_type", "line"),
        }
        queries.append(query)

    payload = {
        "project_id": metadata["project_id"],
        "query_type": metadata.get("data_source_query_type"),
        "time_filters": {
            "bin_size": metadata.get("bin_size", "Day"),
            "time_range": {
                "start_time": metadata["start_time"],
                "end_time": metadata["end_time"],
            },
            "time_zone": metadata.get("time_zone", "UTC"),
        },
        "queries": queries,
    }
    return payload, query_key_map


# Build the appropriate payload based on chart type
ds_type = metadata.get("data_source_query_type")
query_key_map = {}  # maps response query_key -> index into metadata['queries']

if ds_type == "MONITORING":
    query_payload, query_key_map = build_monitoring_payload(metadata)
    query_url = f"{BASE_URL}/v3/queries"
elif ds_type == "ANALYTICS":
    query_payload = build_analytics_payload(metadata)
    query_url = f"{BASE_URL}/v3/analytics/metrics"
elif ds_type in ("GEN_AI_MONITORING", "GEN_AI_METRICS"):
    query_payload, query_key_map = build_genai_monitoring_payload(metadata)
    query_url = f"{BASE_URL}/v3/queries"
else:
    raise ValueError(f"Unsupported chart query type: {ds_type}")

print(f"Query URL: {query_url}")
print(f"Query type: {ds_type}")
print(f"\nPayload:")
print(json.dumps(query_payload, indent=2, default=str))

Query URL: https://demo.fiddler.ai/v3/queries
Query type: MONITORING

Payload:
{
  "project_id": "1eaf9070-af0c-4d50-a3c4-1f7968bd0574",
  "query_type": "MONITORING",
  "filters": {
    "bin_size": "Day",
    "time_zone": "UTC",
    "time_range": {
      "start_time": "2025-09-17T00:00:00",
      "end_time": "2025-10-16T23:59:59"
    },
    "time_label": "30d"
  },
  "queries": [
    {
      "metric": "jsd",
      "model_id": "8dea99c0-6724-46df-a2de-5d542ed7f272",
      "query_key": "5324a767-5467-4d3e-becb-82df91c3a64e",
      "viz_type": "line",
      "columns": [
        "probability_churn"
      ],
      "baseline_id": "5ff6b0a5-c1fc-4620-b262-21d13ad6a51c",
      "segment": {}
    },
    {
      "metric": "jsd",
      "model_id": "8dea99c0-6724-46df-a2de-5d542ed7f272",
      "query_key": "03dc9802-18b8-49a0-b212-5812a6af8e3e",
      "viz_type": "line",
      "columns": [
        "probability_churn"
      ],
      "baseline_id": "5ff6b0a5-c1fc-4620-b262-21d13ad6a51c",
      "segme

In [11]:
# Execute the query
response = requests.post(query_url, headers=HEADERS, json=query_payload)
response.raise_for_status()
query_response = response.json()

print(f"Response status: {response.status_code}")
print(f"Response kind: {query_response.get('kind')}")

Response status: 200
Response kind: NORMAL


---
## Step 4: Parse Results into X, Y Values and Save to CSV

Extract the time-series data points from the response and write them to a CSV file. Each query in the chart produces its own series.

In [13]:
def parse_monitoring_results(query_response, metadata, query_key_map=None):
    """Parse MONITORING or GEN_AI query results into (x, y) rows.

    Args:
        query_response: The JSON response from /v3/queries.
        metadata: Chart metadata dict from extract_chart_metadata().
        query_key_map: Dict mapping each response query_key to the
            index in metadata['queries']. Built by the build_*_payload
            functions. Required for correct metric/model labeling.

    Returns a list of dicts with keys:
        query_index, metric, model, timestamp, value
    """
    if query_key_map is None:
        query_key_map = {}
    rows = []
    results = query_response.get("data", {}).get("results", {})
    queries = metadata.get("queries", [])

    for query_key, series_data in results.items():
        # Look up the original query via the key map from payload construction
        query_info = None
        query_idx = query_key_map.get(query_key)
        if query_idx is not None and query_idx < len(queries):
            query_info = queries[query_idx]

        # The series_data structure varies; handle common formats
        # Format 1: list of [timestamp, value] pairs
        if isinstance(series_data, list):
            for point in series_data:
                if isinstance(point, list) and len(point) >= 2:
                    rows.append({
                        "query_index": query_idx if query_idx is not None else query_key,
                        "metric": (query_info or {}).get("metric") or (query_info or {}).get("metric_name", "unknown"),
                        "model": (query_info or {}).get("model_name") or (query_info or {}).get("model_id", ""),
                        "timestamp": point[0],
                        "value": point[1],
                    })
                elif isinstance(point, dict):
                    rows.append({
                        "query_index": query_idx if query_idx is not None else query_key,
                        "metric": (query_info or {}).get("metric") or (query_info or {}).get("metric_name", "unknown"),
                        "model": (query_info or {}).get("model_name") or (query_info or {}).get("model_id", ""),
                        "timestamp": point.get("time") or point.get("timestamp") or point.get("x"),
                        "value": point.get("value") or point.get("y"),
                    })

        # Format 2: dict with 'values' key containing the series
        elif isinstance(series_data, dict):
            values = series_data.get("values", series_data.get("data", []))
            if isinstance(values, list):
                for point in values:
                    if isinstance(point, list) and len(point) >= 2:
                        rows.append({
                            "query_index": query_idx if query_idx is not None else query_key,
                            "metric": (query_info or {}).get("metric") or (query_info or {}).get("metric_name", "unknown"),
                            "model": (query_info or {}).get("model_name") or (query_info or {}).get("model_id", ""),
                            "timestamp": point[0],
                            "value": point[1],
                        })
                    elif isinstance(point, dict):
                        rows.append({
                            "query_index": query_idx if query_idx is not None else query_key,
                            "metric": (query_info or {}).get("metric") or (query_info or {}).get("metric_name", "unknown"),
                            "model": (query_info or {}).get("model_name") or (query_info or {}).get("model_id", ""),
                            "timestamp": point.get("time") or point.get("timestamp") or point.get("x"),
                            "value": point.get("value") or point.get("y"),
                        })

    return rows


def parse_analytics_results(query_response, metadata):
    """Parse ANALYTICS query results into (x, y) rows.

    Returns a list of dicts with keys:
        metric_id, model, timestamp, value
    """
    rows = []
    data = query_response.get("data", {})

    # Analytics responses may have different structures
    # Common: list of metric results with values
    if isinstance(data, list):
        for item in data:
            metric_id = item.get("id", "unknown")
            value = item.get("value")
            rows.append({
                "metric_id": metric_id,
                "model": metadata.get("model_name") or metadata.get("model_id", ""),
                "timestamp": "",
                "value": value,
            })
    elif isinstance(data, dict):
        # Try to extract from a results key or iterate items
        results = data.get("results", data.get("metrics", data))
        if isinstance(results, dict):
            for key, val in results.items():
                if isinstance(val, (int, float)):
                    rows.append({
                        "metric_id": key,
                        "model": metadata.get("model_name") or metadata.get("model_id", ""),
                        "timestamp": "",
                        "value": val,
                    })
                elif isinstance(val, list):
                    for point in val:
                        if isinstance(point, list) and len(point) >= 2:
                            rows.append({
                                "metric_id": key,
                                "model": metadata.get("model_name") or metadata.get("model_id", ""),
                                "timestamp": point[0],
                                "value": point[1],
                            })

    return rows


# Parse based on chart type
if ds_type in ("MONITORING", "GEN_AI_MONITORING", "GEN_AI_METRICS"):
    rows = parse_monitoring_results(query_response, metadata, query_key_map)
elif ds_type == "ANALYTICS":
    rows = parse_analytics_results(query_response, metadata)
else:
    rows = []

print(f"Parsed {len(rows)} data points")
if rows:
    print(f"\nFirst 5 rows:")
    for row in rows[:5]:
        print(f"  {row}")

Parsed 90 data points

First 5 rows:
  {'query_index': '5324a767-5467-4d3e-becb-82df91c3a64e', 'metric': 'unknown', 'model': '', 'timestamp': '2026-03-31T00:00:00+00:00', 'value': 0.04464257032153243}
  {'query_index': '5324a767-5467-4d3e-becb-82df91c3a64e', 'metric': 'unknown', 'model': '', 'timestamp': '2026-04-01T00:00:00+00:00', 'value': 0.05920637170432753}
  {'query_index': '5324a767-5467-4d3e-becb-82df91c3a64e', 'metric': 'unknown', 'model': '', 'timestamp': '2026-04-02T00:00:00+00:00', 'value': 0.03754789514102216}
  {'query_index': '5324a767-5467-4d3e-becb-82df91c3a64e', 'metric': 'unknown', 'model': '', 'timestamp': '2026-04-03T00:00:00+00:00', 'value': 0.0425610813272346}
  {'query_index': '5324a767-5467-4d3e-becb-82df91c3a64e', 'metric': 'unknown', 'model': '', 'timestamp': '2026-04-04T00:00:00+00:00', 'value': 0.10456980398941924}


In [ ]:
# If parsing returned 0 rows, the response format may not match
# the expected patterns. Inspect the raw response to debug.
if len(rows) == 0:
    print("No rows parsed. Inspecting raw response structure:")
    print(json.dumps(query_response, indent=2, default=str))
    print("\n--- Tip: Examine the structure above and adjust the parsing logic if needed. ---")

In [ ]:
# Save to CSV
if rows:
    # Build a filename from the chart title
    safe_title = metadata["title"].replace(" ", "_").replace("/", "_")[:50]
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"{safe_title}_{timestamp_str}.csv"
    csv_path = os.path.join(OUTPUT_DIR, csv_filename)

    # Determine CSV columns from the first row
    fieldnames = list(rows[0].keys())

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved {len(rows)} rows to: {csv_path}")
    print(f"Columns: {fieldnames}")
else:
    print("No data to save. Check the response parsing above.")

In [ ]:
# Quick preview of the CSV contents
if rows:
    print(f"\n{'='*60}")
    print(f"CSV Preview: {csv_filename}")
    print(f"{'='*60}")
    with open(csv_path, "r") as f:
        for i, line in enumerate(f):
            if i > 10:  # Show header + first 10 data rows
                print(f"  ... ({len(rows) - 10} more rows)")
                break
            print(f"  {line.rstrip()}")

---
## Batch Mode: Pull All Charts from the Dashboard

Optionally loop over all charts in the dashboard, pull metrics for each, and save separate CSV files.

In [ ]:
# Set to True to pull metrics for ALL charts on the dashboard
BATCH_MODE = False

if BATCH_MODE:
    for idx, chart in enumerate(charts):
        chart_meta = extract_chart_metadata(chart)
        chart_ds_type = chart_meta.get("data_source_query_type")

        print(f"\n{'='*60}")
        print(f"[{idx}] {chart_meta['title']} (type: {chart_ds_type})")
        print(f"{'='*60}")

        try:
            # Build payload
            batch_key_map = {}
            if chart_ds_type == "MONITORING":
                payload, batch_key_map = build_monitoring_payload(chart_meta)
                url = f"{BASE_URL}/v3/queries"
            elif chart_ds_type == "ANALYTICS":
                payload = build_analytics_payload(chart_meta)
                url = f"{BASE_URL}/v3/analytics/metrics"
            elif chart_ds_type in ("GEN_AI_MONITORING", "GEN_AI_METRICS"):
                payload, batch_key_map = build_genai_monitoring_payload(chart_meta)
                url = f"{BASE_URL}/v3/queries"
            else:
                print(f"  Skipping unsupported chart type: {chart_ds_type}")
                continue

            # Execute query
            resp = requests.post(url, headers=HEADERS, json=payload)
            resp.raise_for_status()
            resp_json = resp.json()

            # Parse results
            if chart_ds_type in ("MONITORING", "GEN_AI_MONITORING", "GEN_AI_METRICS"):
                chart_rows = parse_monitoring_results(resp_json, chart_meta, batch_key_map)
            elif chart_ds_type == "ANALYTICS":
                chart_rows = parse_analytics_results(resp_json, chart_meta)
            else:
                chart_rows = []

            if chart_rows:
                safe_title = chart_meta["title"].replace(" ", "_").replace("/", "_")[:50]
                ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                fname = f"{safe_title}_{ts}.csv"
                fpath = os.path.join(OUTPUT_DIR, fname)

                fieldnames = list(chart_rows[0].keys())
                with open(fpath, "w", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writeheader()
                    writer.writerows(chart_rows)

                print(f"  Saved {len(chart_rows)} rows -> {fpath}")
            else:
                print(f"  No data points returned.")

        except Exception as e:
            print(f"  ERROR: {e}")

    print(f"\nBatch complete.")
else:
    print("Batch mode disabled. Set BATCH_MODE = True above to pull all charts.")